xarray > numpy > mk tau
https://scipy.github.io/devdocs/reference/generated/scipy.stats.kendalltau.html


In [4]:
# Packages
import os
import xarray as xr
import rioxarray as rxr
import numpy as np
from scipy.stats import kendalltau

In [5]:
# Folder with your TIFF files
files_input = "/home/gisuser/code/data/3_MovingWindow/mw_area"

# Extract year from filename like '1990_area_1km.tif'
def extracted_year(filename):
    base = os.path.basename(filename)
    year_str = base.split('_')[0] # Result: "1990"
    return int(year_str) # returns int

# List TIFF files sorted by year
file_paths = sorted(
    [os.path.join(files_input, f)
    for f in os.listdir(files_input)
    if f.lower().endswith(('.tif', '.tiff'))],
    key=extracted_year
)

In [7]:
# Load files as xarray DataArrays and squeeze band dimension
#arrays = [rxr.open_rasterio(fp, masked=True).squeeze() for fp in file_paths]
arrays = [rxr.open_rasterio(fp, masked=True, chunks={'x':1024, 'y':1024}).squeeze() for fp in file_paths]

In [8]:
# Stack along time dimension (coordinates default 0,1,...)
stacked = xr.concat(arrays, dim='time')

In [11]:
print(stacked['time'])

<xarray.DataArray 'time' (time: 31)> Size: 248B
array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
       18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30])
Coordinates:
    band         int64 8B 1
    spatial_ref  int64 8B 0
Dimensions without coordinates: time


In [15]:
# calulate the number of NaN in a given year to properly set minimum number of years with data present to be counted towards mk tau

# Assume 'stacked' is your xarray.DataArray with dims ('time', 'y', 'x')
total_years = stacked.sizes['time']

# Count number of NaNs per pixel across time
nan_count = stacked.isnull().sum(dim='time')

# Mask pixels where all values are NaN (nan_count == total_years)
valid_pixels_mask = nan_count < total_years

# Calculate valid years count for pixels that have any data
valid_count = total_years - nan_count

# Apply mask to exclude pixels with all NaNs for min and mean calculations
valid_count_masked = valid_count.where(valid_pixels_mask)

# Compute statistics ignoring fully missing pixels
max_missing = nan_count.where(valid_pixels_mask).max().compute().item()
min_valid = valid_count_masked.min().compute().item()
mean_valid = valid_count_masked.mean().compute().item()

print(f"Maximum number of missing years (excluding fully missing pixels): {max_missing}")
print(f"Minimum number of represented years (excluding fully missing pixels): {min_valid}")
print(f"Mean number of represented years per pixel (excluding fully missing pixels): {mean_valid:.2f}")

Maximum number of missing years (excluding fully missing pixels): 0.0
Minimum number of represented years (excluding fully missing pixels): 31.0
Mean number of represented years per pixel (excluding fully missing pixels): 31.00


In [16]:
# Define Mann-Kendall Tau calculation using scipy's kendalltau
# uses numpy (only works in numpy)
# If enough valid data exists (≥3 time points), the output pixel gets its trend tau value.
# If not enough data points remain, the pixel in the output image will be NaN — indicating no reliable trend could be calculated for that pixel.
# Other pixels with sufficient data are unaffected; their tau values are calculated independently.
def mk_tau(arr):
    arr = arr[~np.isnan(arr)]  # Filter NaNs
    if len(arr) < 3:
        return np.nan
    tau, _ = kendalltau(np.arange(len(arr)), arr)
    return tau

# converts to numpy and:
# Applies pixel-wise along time axis (axis=0)
tau_vals = np.apply_along_axis(mk_tau, axis=0, arr=stacked.values)

# Save output Tau raster using metadata from first file
first_raster = arrays[0]
transform = first_raster.rio.transform()
height, width = tau_vals.shape

out_meta = {
    "driver": "GTiff",
    "height": height,
    "width": width,
    "count": 1,
    "dtype": "float32",
    "crs": first_raster.rio.crs,
    "transform": transform
}

output_path = "mann_kendall_tau_result.tif"
with rasterio.open(output_path, "w", **out_meta) as dst:
    dst.write(tau_vals.astype(np.float32), 1)

print(f"Mann-Kendall Tau image saved to {output_path}")

: 

: 

: 

In [ ]:
# Run Code